# Leaflet cluster map of talk locations

Assuming you are working in a Linux or Windows Subsystem for Linux environment, you may need to install some dependencies. Assuming a clean installation, the following will be needed:

```bash
sudo apt install jupyter
sudo apt install python3-pip
pip install python-frontmatter getorg --upgrade
```

After which you can run this from the `_talks/` directory, via:

```bash
 jupyter nbconvert --to notebook --execute talkmap.ipynb --output talkmap_out.ipynb
```
 
The `_talks/` directory contains `.md` files of all your talks. This scrapes the location YAML field from each `.md` file, geolocates it with `geopy/Nominatim`, and uses the `getorg` library to output data, HTML, and Javascript for a standalone cluster map.

In [1]:
# Start by installing the dependencies
!pip install python-frontmatter getorg --upgrade
import frontmatter
import glob
import getorg
from geopy import Nominatim
from geopy.exc import GeocoderTimedOut

  Using cached pynacl-1.6.2-cp38-abi3-macosx_10_10_universal2.whl.metadata (10.0 kB)


  Using cached pyjwt-2.13.0-py3-none-any.whl.metadata (3.4 kB)


  Using cached cryptography-49.0.0-cp39-abi3-macosx_11_0_arm64.whl.metadata (4.3 kB)


Using cached pyjwt-2.13.0-py3-none-any.whl (31 kB)
Using cached cryptography-49.0.0-cp39-abi3-macosx_11_0_arm64.whl (4.0 MB)


Using cached pynacl-1.6.2-cp38-abi3-macosx_10_10_universal2.whl (388 kB)


...

      Successfully uninstalled cffi-1.17.1
   ━━━━━━━━━━━━━━━━╺━━━━━━━━━━━━━━━━━━━━━━━  4/10 [cffi]

   ━━━━━━━━━━━━━━━━━━━━━━━━╺━━━━━━━━━━━━━━━  6/10 [geopy]

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╺━━━━━━━  8/10 [pygithub]

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10/10 [getorg]


Iywidgets and ipyleaflet support disabled. You must be in a Jupyter notebook to use this feature.
Error raised:
No module named 'ipyleaflet'
Check that you have enabled ipyleaflet in Jupyter with:
    jupyter nbextension enable --py ipyleaflet


In [2]:
# Collect the Markdown files
g = glob.glob("_talks/*.md")

In [3]:
# Set the default timeout, in seconds
TIMEOUT = 5

# Prepare to geolocate
geocoder = Nominatim(user_agent="academicpages.github.io")
location_dict = {}
location = ""
permalink = ""
title = ""

In the event that this times out with an error, double check to make sure that the location is can be properly geolocated.

In [4]:
# Perform geolocation
for file in g:
    # Read the file
    data = frontmatter.load(file)
    data = data.to_dict()

    # Press on if the location is not present
    if 'location' not in data:
        continue

    # Prepare the description
    title = data['title'].strip()
    venue = data['venue'].strip()
    location = data['location'].strip()
    description = f"{title}<br />{venue}; {location}"

    # Geocode the location and report the status
    try:
        location_dict[description] = geocoder.geocode(location, timeout=TIMEOUT)
        print(description, location_dict[description])
    except ValueError as ex:
        print(f"Error: geocode failed on input {location} with message {ex}")
    except GeocoderTimedOut as ex:
        print(f"Error: geocode timed out on input {location} with message {ex}")
    except Exception as ex:
        print(f"An unhandled exception occurred while processing input {location} with message {ex}")

ShapBPT in Perspective: A Consolidated Review and an eXplainable Anomaly Detection Case Study<br />QualITA Workshop | ICPE 2026; Florence, Italy Firenze, Toscana, Italia


Can I Trust My Anomaly Detection System? A Case Study Based on Explainable AI<br />Mediterranean Conference Centre; La Valletta, Malta Il-Belt Valletta, Reġjun tal-Port, Malta


Using Stratified Sampling to Improve LIME Image Explanations<br />Vancouver Convention Centre; Vancouver, Canada Vancouver, Metro Vancouver Regional District, British Columbia, Canada


ShapBPT: Image Feature Attributions using Data-Aware Binary Partition Trees<br />Singapore Expo Centre; Singapore Singapore


Unexpected Condition Detector for Industrial Safety using Deep Generative Models<br />Universidad de Vigo; Vigo, Spain Vigo, Pontevedra, Galicia, España
Improving Trust in Safety-Critical AI Systems: Explainable AI and Anomaly Detection Frameworks for Human Safety in Smart Industries<br />University of Torino; Turin, Italy Torino, Piemonte, Italia


In [5]:
# Save the map
m = getorg.orgmap.create_map_obj()
getorg.orgmap.output_html_cluster_map(location_dict, folder_name="talkmap", hashed_usernames=False)

'Written map to talkmap/'